# 第5回　相関と因果
## ―― 「相関がある」と「原因である」は、まったく違う

統計学Ⅰ（B）　／　北星学園大学

今日は**▶を上から押す**。ところどころ「やってみよう」で列名を変えて試せる。注目は ――

> 2つのものが一緒に動いても、**一方が他方の原因とは限らない。**

In [ ]:
!pip install -q japanize-matplotlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import japanize_matplotlib  # noqa: F401

URL = "https://aonoa68.github.io/toukei-1/data/hokushin_students.csv"
try:
    df = pd.read_csv(URL)
except Exception:
    R = np.random.default_rng(2026); N = 400
    disc = R.normal(0,1,N); apt = R.normal(0,1,N)
    gk = R.choice(["経済学部","文学部","社会福祉学部"], N, p=[.40,.35,.25])
    gke = np.select([gk=="経済学部",gk=="文学部",gk=="社会福祉学部"],[2.,-1.,-1.])
    gen = R.choice(["女","男","回答しない"], N, p=[.55,.43,.02])
    bh = np.where(gen=="男",171.,158.); bh=np.where(gen=="回答しない",165.,bh)
    height = bh + R.normal(0,6,N)
    alone = R.random(N)<.35
    com = np.clip(np.where(alone,R.normal(20,8,N),R.normal(55,25,N)),5,None)
    slp = 7+.5*disc-.005*(com-30)+R.normal(0,.8,N)
    sns = np.clip(3-.8*disc+R.normal(0,1.,N),0,None)
    std = np.clip(1.5+.7*disc+.2*apt+R.normal(0,.6,N),0,None)
    pt  = np.clip(np.where(alone,R.normal(18,6,N),R.normal(10,6,N)),0,None)
    att = np.clip(82+7*disc+R.normal(0,5,N),0,100)
    bf  = np.clip(np.round(4+1.6*disc+R.normal(0,1.,N)),0,7)
    test= np.clip(55+7*apt+4*std+1.2*(slp-7)-1.5*sns+gke+R.normal(0,6,N),0,100)
    inc = R.lognormal(np.log(550),.45,N)
    df = pd.DataFrame({"学生ID":[f"26B{i+1:04d}" for i in range(N)],"学部":gk,"性別":gen,
        "身長cm":np.round(height,1),"一人暮らし":np.where(alone,"はい","いいえ"),
        "通学時間min":np.round(com).astype(int),"睡眠時間h":np.round(slp,1),"SNS時間h":np.round(sns,1),
        "勉強時間h":np.round(std,1),"アルバイト時間week":np.round(pt).astype(int),"出席率":np.round(att).astype(int),
        "朝食日数week":bf.astype(int),"テスト点":np.round(test).astype(int),"世帯年収万円":np.round(inc).astype(int)})
    df.loc[3,"世帯年収万円"]=12000; df.loc[88,"世帯年収万円"]=9500; df.loc[7,"身長cm"]=1710.0
    df.loc[15,"睡眠時間h"]=np.nan; df.loc[42,"睡眠時間h"]=np.nan; df.loc[101,"通学時間min"]=np.nan
df = df.dropna()
print("準備OK")

---
## フック：この相関を信じる？

- 「**アイスの売上**が増えると、**水難事故**も増える」――強い相関がある。ではアイスを禁止すれば事故は減る？
- 「**チョコの消費量**が多い国ほど**ノーベル賞**受賞者が多い」――チョコを食べれば頭が良くなる？

どちらも「相関」は本物。でも結論はばかげている。**なぜか**を、今日のデータで解き明かす。

---
## 1. 散布図と相関係数 r

2つの量の関係は、**散布図**（点を打つ）で見るのが基本。その関係の強さを1つの数にしたのが **相関係数 r**（−1〜+1）。

- r が +1 に近い：片方が増えると、もう片方も増える（右上がり）
- r が −1 に近い：片方が増えると、もう片方は減る（右下がり）
- r が 0 に近い：直線的な関係がない

まず「勉強時間 と テスト点」を見てみよう。

In [ ]:
r = df["勉強時間h"].corr(df["テスト点"])
print(f"勉強時間 と テスト点 の相関係数 r = {r:.3f}")

plt.figure(figsize=(6,4))
plt.scatter(df["勉強時間h"], df["テスト点"], s=12, alpha=0.5, color="#00897b")
plt.xlabel("勉強時間h"); plt.ylabel("テスト点"); plt.title(f"勉強するほど点が高い（r={r:.2f}）")
plt.show()

r ≈ 0.60。右上がりで、勉強時間が長い人ほどテスト点が高い。これは**納得のいく**関係だ。

では次の相関はどうだろう？

---
## 2. 「朝食を食べると成績が上がる」？

`朝食日数week`（週に何日朝食を食べたか）と `テスト点` の相関を見る。

In [ ]:
r = df["朝食日数week"].corr(df["テスト点"])
print(f"朝食日数 と テスト点 の相関係数 r = {r:.3f}")

plt.figure(figsize=(6,4))
plt.scatter(df["朝食日数week"], df["テスト点"], s=12, alpha=0.4, color="#e8503a")
plt.xlabel("朝食日数 / 週"); plt.ylabel("テスト点"); plt.title(f"朝食をよく食べる人ほど点が高い？（r={r:.2f}）")
plt.show()

r ≈ 0.37。「**朝食を食べれば成績が上がる**」と言いたくなる。新聞の見出しになりそうだ。

でも、本当に**朝食が原因**なのだろうか？　 ―― ここで立ち止まるのが統計の思考だ。

---
## 3. 交絡を疑え ―― 第三の変数

「朝食をきちんと食べる人」は、たぶん**生活がきちんとしている**。ということは、**よく勉強し、よく寝ている**かもしれない。確かめよう。

In [ ]:
print(f"朝食日数 と 勉強時間 の相関 r = {df['朝食日数week'].corr(df['勉強時間h']):.3f}")
print("→ 朝食をよく食べる人は、よく勉強もしている（生活の規律が両方を動かしている）")

朝食と勉強は r≈0.60 で強く結びついている。つまり――

```
   （生活の規律）
      ／      ＼
   朝食       勉強・睡眠 ──→ テスト点
```

本当にテスト点を動かしているのは**勉強・睡眠**で、朝食は「規律の高い人がついでにやっていること」かもしれない。これを **交絡（こうらく）** という。

確かめ方：**勉強・睡眠の影響を取り除いて**、朝食とテスト点の関係が残るか見る（偏相関）。

In [ ]:
# 勉強・睡眠の効果を引き算してから、朝食とテスト点の相関を測る
def 偏相関(y, x, 統制):
    X = np.column_stack([np.ones(len(df))] + [df[c] for c in 統制])
    def 残差(v):
        b = np.linalg.lstsq(X, df[v], rcond=None)[0]
        return df[v] - X @ b
    return np.corrcoef(残差(x), 残差(y))[0, 1]

print(f"朝食とテスト点：単純な相関       r = {df['朝食日数week'].corr(df['テスト点']):.3f}")
print(f"朝食とテスト点：勉強・睡眠を統制後 r = {偏相関('テスト点','朝食日数week',['勉強時間h','睡眠時間h']):.3f}")
print("\n→ ほぼ 0。朝食とテスト点の見かけの関係は、勉強・睡眠（＝規律）による交絡だった。")

**朝食の効果は消えた。** アイスと水難事故（犯人は「気温」）、チョコとノーベル賞（犯人は「経済的豊かさ」）と同じ構造だ。

> **相関を見たら、因果を結論する前に『隠れた第三の変数（交絡）』を疑え。**

---
## 4. r だけ見るな ―― アンスコムの四重奏

有名な例。**4つのまったく違うデータ**が、どれも r = 0.816、回帰直線もほぼ同じ。でも散布図は別物。

In [ ]:
x = np.array([10,8,13,9,11,14,6,4,12,7,5], float)
x4 = np.array([8,8,8,8,8,8,8,19,8,8,8], float)
ys = {
  "I 直線的":  [8.04,6.95,7.58,8.81,8.33,9.96,7.24,4.26,10.84,4.82,5.68],
  "II 曲線":   [9.14,8.14,8.74,8.77,9.26,8.10,6.13,3.10,9.13,7.26,4.74],
  "III 外れ値":[7.46,6.77,12.74,7.11,7.81,8.84,6.08,5.39,8.15,6.42,5.73],
  "IV 1点が支配":[6.58,5.76,7.71,8.84,8.47,7.04,5.25,12.50,5.56,7.91,6.89],
}
fig, ax = plt.subplots(1, 4, figsize=(13, 3.2))
for i,(name,y) in enumerate(ys.items()):
    xx = x4 if name.startswith("IV") else x
    rr = np.corrcoef(xx, y)[0,1]
    ax[i].scatter(xx, y, color="#00897b"); ax[i].set_title(f"{name}\nr={rr:.2f}")
    ax[i].set_ylim(2,13); ax[i].set_xlim(3,20)
plt.tight_layout(); plt.show()

全部 r=0.82。でも――IIは曲線、IIIは1個の外れ値、IVはたった1点が全体を支配している。**r という1つの数だけ見て『関係あり』と報告するのは危険。必ず散布図を見る。**

---
## 5. r の限界：曲がった関係は見えない

In [ ]:
xx = np.linspace(-3, 3, 100)
yy = xx**2          # きれいな放物線（強い関係）
print(f"放物線 y=x^2 の相関係数 r = {np.corrcoef(xx, yy)[0,1]:.3f}（ほぼ0！）")
plt.figure(figsize=(5,3.5))
plt.scatter(xx, yy, s=10, color="#00897b")
plt.title("明らかに関係あり。でも r ≈ 0"); plt.xlabel("x"); plt.ylabel("y")
plt.show()

r が測れるのは**直線的な**関係だけ。U字やくの字の関係は r ≈ 0 になり、「関係なし」と誤解してしまう。やはり**まず散布図**。

---
## 今日のまとめ

| 注意 | 中身 |
|---|---|
| 相関 ≠ 因果 | 一緒に動いても原因とは限らない |
| 交絡 | 隠れた第三の変数が両方を動かす（朝食の例＝規律） |
| 逆因果 | 原因と結果が逆かも |
| 偶然 | たまたま相関することもある |
| r だけ見ない | アンスコム・非線形・外れ値 → 必ず散布図 |

> **相関を見たら、まず散布図を描き、次に『第三の変数』を疑う。**
> 因果を確かめるには、本当は**実験（ランダム化）**が要る ―― それは後の回で。

**課題（Moodle）**：与えられた相関事例について「因果と言えるか」「対抗仮説（交絡など）を1つ」を答える。